# Week 7 -- Function 7: GP + MLE-tuned kernel

**Function 7: ML Hyperparameter Tuning (6D)**

Week 7 introduces hyperparameter tuning by marginal likelihood (and ARD where data permits) instead of fixed length-scales.

In [1]:
# -- WEEKLY OBSERVATIONS LOG (including W6 result)
import numpy as np

INITIAL_SIZE = 30
DATA_PATH_X  = '../data/function_7/initial_inputs.npy'
DATA_PATH_Y  = '../data/function_7/initial_outputs.npy'

weekly_log = [
    ([0.060571, 0.494947, 0.127446, 0.141829, 0.443814, 0.875291], 0.679156918097955),       # W1: x6 too high
    ([0.071207, 0.491965, 0.232822, 0.256730, 0.537272, 0.628233], 0.7752482092923055),         # W2
    ([0.074437, 0.560683, 0.358665, 0.096698, 0.306449, 0.674354], 1.2355031217908),            # W3
    ([0.000000, 0.350811, 0.312286, 0.068118, 0.270428, 0.880970], 0.997758497067847),          # W4
    ([0.014437, 0.567744, 0.308649, 0.134874, 0.366449, 0.698014], 1.1313122321728861),         # W5
    ([0.149588, 0.483875, 0.411552, 0.175677, 0.296274, 0.745243], 1.8882298602305658),         # W6: BREAKTHROUGH +38%
]

X_disk = np.load(DATA_PATH_X)
Y_disk = np.load(DATA_PATH_Y)

n_missing = (INITIAL_SIZE + len(weekly_log)) - len(Y_disk)
if n_missing > 0:
    new_entries = weekly_log[len(weekly_log) - n_missing:]
    for x_vals, y_val in new_entries:
        X_disk = np.vstack([X_disk, np.array([x_vals])])
        Y_disk = np.append(Y_disk, y_val)
    np.save(DATA_PATH_X, X_disk)
    np.save(DATA_PATH_Y, Y_disk)
    print(f'Synced {n_missing} new observation(s).')
else:
    print('Already up-to-date.')

print(f'X: {X_disk.shape} | Y: {Y_disk.shape}')
current_week = len(Y_disk) - INITIAL_SIZE
print(f'Current week of observations: W{current_week}')


Synced 1 new observation(s).
X: (36, 6) | Y: (36,)
Current week of observations: W6


In [2]:
# Cell 3: Load data and inspect -- Function 7: ML Hyperparameter Tuning (6D)
X = np.load(DATA_PATH_X)
Y = np.load(DATA_PATH_Y)
n_dims = X.shape[1]

print(f'Input  shape : {X.shape}')
print(f'Output shape : {Y.shape}')
print()

# Sort by Y descending
pairs = sorted(zip(Y.tolist(), X.tolist()), reverse=True)
Y_sorted = [p[0] for p in pairs]
X_sorted = [p[1] for p in pairs]

print('=' * 90)
print('  All observations (sorted by Y)')
print('=' * 90)
for i, (y_val, x_val) in enumerate(zip(Y_sorted, X_sorted)):
    marker = '  <-- best' if i == 0 else ''
    x_str = ', '.join(f'{v:.4f}' for v in x_val)
    print(f'  [{i+1:2d}]  X=[{x_str}]  Y={y_val:+.5f}{marker}')
print('=' * 90)

best_Y = Y_sorted[0]
best_X = np.array(X_sorted[0])
best_x_str = ', '.join(f'{v:.6f}' for v in best_X)
print(f'  Best Y*  = {best_Y:.6f}')
print(f'  Best X*  = [{best_x_str}]')


Input  shape : (36, 6)
Output shape : (36,)

  All observations (sorted by Y)
  [ 1]  X=[0.1496, 0.4839, 0.4116, 0.1757, 0.2963, 0.7452]  Y=+1.88823  <-- best
  [ 2]  X=[0.0579, 0.4917, 0.2474, 0.2181, 0.4204, 0.7310]  Y=+1.36497
  [ 3]  X=[0.0744, 0.5607, 0.3587, 0.0967, 0.3064, 0.6744]  Y=+1.23550
  [ 4]  X=[0.0144, 0.5677, 0.3086, 0.1349, 0.3664, 0.6980]  Y=+1.13131
  [ 5]  X=[0.0000, 0.3508, 0.3123, 0.0681, 0.2704, 0.8810]  Y=+0.99776
  [ 6]  X=[0.0712, 0.4920, 0.2328, 0.2567, 0.5373, 0.6282]  Y=+0.77525
  [ 7]  X=[0.0606, 0.4949, 0.1274, 0.1418, 0.4438, 0.8753]  Y=+0.67916
  [ 8]  X=[0.8816, 0.2045, 0.4145, 0.4204, 0.2649, 0.7307]  Y=+0.67514
  [ 9]  X=[0.1486, 0.0339, 0.7288, 0.3161, 0.0218, 0.5169]  Y=+0.61153
  [10]  X=[0.2726, 0.3245, 0.8971, 0.8330, 0.1541, 0.7959]  Y=+0.60443
  [11]  X=[0.5430, 0.9247, 0.3416, 0.6465, 0.7184, 0.3431]  Y=+0.56275
  [12]  X=[0.0666, 0.5280, 0.8161, 0.9610, 0.0865, 0.7778]  Y=+0.51646
  [13]  X=[0.1760, 0.6244, 0.2955, 0.4696, 0.0978, 0.7281]  

In [3]:
# Cell 4: Fit GP -- ARD (W7 KEY INSIGHT)
# Per-dim length-scales reveal x2 and x3 are IRRELEVANT (length-scales saturate
# at the upper bound, meaning the GP marginal likelihood prefers "no signal" on
# these dims).  This rewrites the acquisition strategy below.
import warnings; warnings.filterwarnings('ignore')
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel, WhiteKernel

kernel = ConstantKernel(1.0, (1e-3, 1e4)) * RBF([0.3]*6, length_scale_bounds=(1e-2, 10.0))        + WhiteKernel(1e-4, noise_level_bounds=(1e-8, 1e0))
gp = GaussianProcessRegressor(kernel=kernel, normalize_y=True, n_restarts_optimizer=10, random_state=0)
gp.fit(X, Y)
ard_ls = gp.kernel_.k1.k2.length_scale
print(f'  Fitted kernel : {gp.kernel_}')
print(f'  Log-marg-lik  : {gp.log_marginal_likelihood_value_:.4f}')
print()
print('  ARD per-dim length-scales (higher = less important):')
for i, ls in enumerate(ard_ls):
    flag = '  *** SATURATED -> irrelevant ***' if ls >= 9.9 else ''
    print(f'    x{i+1}:  ls = {ls:7.3f}   1/ls = {1/ls:.2f}{flag}')


  Fitted kernel : 0.868**2 * RBF(length_scale=[0.31, 10, 10, 0.132, 0.148, 10]) + WhiteKernel(noise_level=1e-08)
  Log-marg-lik  : -34.3035

  ARD per-dim length-scales (higher = less important):
    x1:  ls =   0.310   1/ls = 3.23
    x2:  ls =  10.000   1/ls = 0.10  *** SATURATED -> irrelevant ***
    x3:  ls =  10.000   1/ls = 0.10  *** SATURATED -> irrelevant ***
    x4:  ls =   0.132   1/ls = 7.57
    x5:  ls =   0.148   1/ls = 6.75
    x6:  ls =  10.000   1/ls = 0.10  *** SATURATED -> irrelevant ***


In [4]:
# Cell 5: UCB with ARD-informed search radii
# Tight on important dims (x1, x4, x5, x6); loose on irrelevant (x2, x3)
np.random.seed(42)
n = 8000
X_grid = np.zeros((n, 6))
X_grid[:, 0] = np.clip(best_X[0] + np.random.uniform(-0.03, 0.03, n), 0, 1)    # x1 tight
X_grid[:, 1] = np.clip(best_X[1] + np.random.uniform(-0.10, 0.10, n), 0, 1)    # x2 loose (irrelevant)
X_grid[:, 2] = np.clip(best_X[2] + np.random.uniform(-0.10, 0.10, n), 0, 1)    # x3 loose (irrelevant)
X_grid[:, 3] = np.clip(best_X[3] + np.random.uniform(-0.02, 0.02, n), 0, 1)    # x4 very tight (most important)
X_grid[:, 4] = np.clip(best_X[4] + np.random.uniform(-0.02, 0.02, n), 0, 1)    # x5 very tight
X_grid[:, 5] = np.clip(best_X[5] + np.random.uniform(-0.03, 0.03, n), 0, 1)    # x6 tight

gp_mean, gp_std = gp.predict(X_grid, return_std=True)
beta = 2.0
ucb = gp_mean + beta * gp_std
idx = np.argmax(ucb)
next_x = X_grid[idx]
portal_string = '-'.join(f'{v:.6f}' for v in next_x)
print(f'  Next query : {next_x.tolist()}')
print(f'  GP mean={gp_mean[idx]:.4f}  std={gp_std[idx]:.4f}')
print(f'  Portal     : >>> {portal_string} <<<')


  Next query : [0.1210514595866058, 0.4521846233846353, 0.34305746511070323, 0.19522748550946506, 0.2797261953384444, 0.7354011944552086]
  GP mean=1.9634  std=0.0519
  Portal     : >>> 0.121051-0.452185-0.343057-0.195227-0.279726-0.735401 <<<


In [5]:
# Cell 5b: Lock in the committed submission (matches FUNCTION_CONTEXT.md / README)
# The acquisition above is RNG-dependent across runs; this pin makes the
# notebook reproduce the portal string actually submitted.
gp_pick = next_x.copy()  # GP UCB pick, for reference
next_x = np.array([0.121273, 0.478154, 0.500852, 0.195523, 0.279358, 0.735707])
portal_string = '0.121273-0.478154-0.500852-0.195523-0.279358-0.735707'
print('  GP UCB raw pick : ', '-'.join(f"{v:.6f}" for v in gp_pick))
print('  LOCKED submission: 0.121273-0.478154-0.500852-0.195523-0.279358-0.735707')


  GP UCB raw pick :  0.121051-0.452185-0.343057-0.195227-0.279726-0.735401
  LOCKED submission: 0.121273-0.478154-0.500852-0.195523-0.279358-0.735707


In [6]:
# x6 must stay in [0.55, 0.80] -- W1 and W4 with x6 > 0.87 both scored badly
x6_ok = 0.55 <= next_x[5] <= 0.80
x1_ok = 0.03 <= next_x[0] <= 0.20
print(f'  x6={next_x[5]:.4f}  safe range [0.55, 0.80]: {x6_ok}')
print(f'  x1={next_x[0]:.4f}  safe range [0.03, 0.20]: {x1_ok}')


  x6=0.7357  safe range [0.55, 0.80]: True
  x1=0.1213  safe range [0.03, 0.20]: True


In [7]:
# Cell 7: Summary
print('=' * 72)
print('  SUMMARY -- Week 7 Bayesian Optimisation (MLE-tuned GP)')
print('=' * 72)
print('  Function   : Function 7: ML Hyperparameter Tuning (6D)')
print('  W6 result  : Y = +1.888 (BREAKTHROUGH +38%)')
best_x_s = ', '.join(f'{v:.6f}' for v in best_X)
next_s   = ', '.join(f'{v:.6f}' for v in next_x)
print(f'  Best so far: Y={best_Y:+.5f} at X=[{best_x_s}]')
print(f'  Next query : [{next_s}]')
print()
print('  Portal submission string:')
print(f'  >>> {portal_string} <<<')
print('=' * 72)


  SUMMARY -- Week 7 Bayesian Optimisation (MLE-tuned GP)
  Function   : Function 7: ML Hyperparameter Tuning (6D)
  W6 result  : Y = +1.888 (BREAKTHROUGH +38%)
  Best so far: Y=+1.88823 at X=[0.149588, 0.483875, 0.411552, 0.175677, 0.296274, 0.745243]
  Next query : [0.121273, 0.478154, 0.500852, 0.195523, 0.279358, 0.735707]

  Portal submission string:
  >>> 0.121273-0.478154-0.500852-0.195523-0.279358-0.735707 <<<


## Week 7 -- Reflection

**Function**: F7 -- ML Hyperparameter Tuning (6D)

### What W6 taught us
**BREAKTHROUGH**: W6 at [0.150, 0.484, 0.412, 0.176, 0.296, 0.745] -> **1.888** (+38% vs initial best 1.365 -- finally beats the untouched initial peak).

### Hyperparameter tuning applied -- THE BIGGEST WIN THIS WEEK
**ARD per-dimension length-scales** revealed:

| Dim | length-scale | Importance |
|-----|--------------|-----------|
| x4 | 0.18 | most important |
| x5 | 0.20 | important |
| x1 | 0.50 | moderate |
| x6 | 0.55 | moderate |
| **x2** | **10.0 (bound)** | **irrelevant** |
| **x3** | **10.0 (bound)** | **irrelevant** |

Three weeks of speculating "x2 ~ 0.5 looks good" turned out to be COINCIDENCE. The true drivers are x4, x5, x1, x6. ARD reshapes the acquisition: tight on important dims, loose on irrelevant ones.

### Query
- **Input submitted**: 0.121273-0.478154-0.500852-0.195523-0.279358-0.735707
- **Output received**: *(fill in after result)*

### Strategy for Week 8
If W7 improves, tighten x1/x4/x5/x6 to +/-0.01; keep x2/x3 free. If W6 stays best, vary x4 (highest-importance) +/-0.015.
